In [ ]:
from pathlib import Path
from skimage.io import imread
from skimage.measure import regionprops_table
import pandas as pd

import napari

In [ ]:
in_path = '/Volumes/nn/Julia Vogtmann/Microscopy/25AM11-03_4'

mask_outer_subdirectory = 'segmentation_nuclei_edgesnap'
mask_inner_subdirectory = 'segmentation_nucleoli_edgesnap'

out_subdirectory = 'nested_regionprops'

regionprops_to_measure = ('area', )

In [ ]:
mask_files_outer = sorted((Path(in_path) / mask_outer_subdirectory).glob('[!.]*.tif'))
mask_files_inner = sorted((Path(in_path) / mask_inner_subdirectory).glob('[!.]*.tif'))

mask_files_outer, mask_files_inner

In [ ]:
out_path = Path(in_path) / out_subdirectory
if not out_path.exists():
    out_path.mkdir()

for mask_file_outer, mask_file_inner in zip(mask_files_outer, mask_files_inner):

    # load masks
    mask_outer = imread(mask_file_outer)
    mask_inner = imread(mask_file_inner)
    
    # TEST: make outer mask binary
    # mask_outer = (mask_outer > 0).astype(int)
    
    # treat inner mask as binary semantic
    # keep only intersection with outer, assign label of outer
    mask_inner = mask_inner > 0
    mask_inner = (mask_inner & (mask_outer > 0)).astype(int)
    mask_inner *= mask_outer
    
    # regionprops tables -> join
    df_inner = pd.DataFrame(regionprops_table(mask_inner, properties=('label',) + regionprops_to_measure))
    df_outer = pd.DataFrame(regionprops_table(mask_outer, properties=('label',) + regionprops_to_measure))
    df = df_outer.merge(df_inner, on='label', suffixes=('_outer', '_inner'))
    
    # add mask files used (relative to base path)
    df['mask_file_outer'] = Path(mask_file_outer).relative_to(in_path)
    df['mask_file_inner'] = Path(mask_file_inner).relative_to(in_path)
    
    # save (base name on outer mask)
    out_file = out_path / (mask_file_outer.stem + '_nested_rprops.csv')
    df.to_csv(out_file, index=None)